## Libraries

In [4]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## Hugging Face login

In [ ]:
from huggingface_hub import login
login("hf_********")

In [6]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [7]:
!nvidia-smi

Tue Mar 17 13:17:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [10]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Load Dataset

In [ ]:
!ls /kaggle/input

In [ ]:
!ls /kaggle/input/datasets/bengali-empathetic-conversations-corpus

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [13]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/datasets/raseluddin/bengali-empathetic-conversations-corpus/BengaliEmpatheticConversationsCorpus .csv"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [14]:
print(dataset)
print(dataset["train"].column_names)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['Topics', 'Question-Title', 'Questions', 'Answers'],
        num_rows: 38233
    })
})
['Topics', 'Question-Title', 'Questions', 'Answers']
{'Topics': 'পারিবারিক দ্বন্দ্ব', 'Question-Title': 'মা ও স্ত্রীর মধ্যে মতানৈক্য বৃদ্ধি', 'Questions': ' আমার স্ত্রী এবং মায়ের মধ্যে টানটান মতবিরোধ চলছে। অতীতে, তাদের মধ্যে ছোটখাটো পার্থক্য ছিল। উদাহরণস্বরূপ, আমার স্ত্রী আমার কাছে অভিযোগ করবে যে আমার মা খুব কর্তৃত্বপ্রয়াসী; আমার মা অভিযোগ করবেন আমার স্ত্রী অলস। তবে ইদানীং তা তীব্রতর হয়েছে । আমি মনে করি, এর কারণ হচ্ছে আমার স্ত্রী তার সাথে একবার কথার প্রতিত্তর করেছিল। এখন, যেকোনো সামান্য মতবিরোধকে বড় করা হয়, যা বড় মতবিরোধের দিকে পরিচালিত করে। আমি কি করতে পারি?', 'Answers': ' আপনি যা বর্ণনা করছেন তাকে মনোবিজ্ঞানীরা "ত্রিভুজকরণ" বলে অভিহিত করেছেন। যা হয় যখন পরিবারের একজন সদস্য যার সাথে তাদের সমস্যা আছে তার সাথে কথা না বলে এবং পরিবর্তে পরিবারের তৃতীয় সদস্যের কাছে অভিযোগ জানাতে যায়। আপনাকে \'ত্রিভুজাকার\' করা হয়েছে; আপনার স্ত্রী এবং মায়ের দ্

## LLAMAFineTuner

In [18]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

class LLAMAFineTuner:

    def __init__(self, model_name, max_seq_length=8192):
        self.model_name = model_name
        self.max_seq_length = max_seq_length
        self.model = None
        self.tokenizer = None
        self.trainer = None

    def load_model(self):

        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=self.model_name,
            max_seq_length=self.max_seq_length,
            dtype=None,
            load_in_4bit=True,
            device_map={"": 0}
        )

    def apply_lora(self):

        self.model = FastLanguageModel.get_peft_model(
            self.model,
            r=16,
            target_modules=[
                "q_proj","k_proj","v_proj","o_proj",
                "gate_proj","up_proj","down_proj"
            ],
            lora_alpha=32,
            lora_dropout=0,
            bias="none",
            use_gradient_checkpointing=True,
            random_state = 3407
        )

    def train(self, train_dataset, val_dataset):

        self.trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            dataset_text_field="text",
            max_seq_length=self.max_seq_length,
            args=TrainingArguments(
                per_device_train_batch_size=1,
                gradient_accumulation_steps=16,
                learning_rate=2e-4,
                max_steps=200,
                fp16=True,
                logging_steps=10,
                output_dir="outputs",
                save_strategy="steps",
                save_steps=50,
                save_total_limit=2
            )
        )

        self.trainer.train()

    def get_trainer(self):
        return self.trainer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-03-17 13:20:58.796152: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773753658.991960      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773753659.048009      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773753659.524554      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773753659.524593      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773753659.524595      55 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using

In [24]:
tuner = LLAMAFineTuner(
    model_name="unsloth/llama-3.1-8b-instruct",
    max_seq_length=8192
)

tuner.load_model()   # loads 4bit model
tuner.apply_lora()   # applies LoRA


==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## DatasetProcessor

In [28]:
class DatasetProcessor:

    def __init__(self, tokenizer, max_length=8192):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def format_prompt(self, example):

        text = f"""<|begin_of_text|><|system|>
আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকারীর অনুভূতি বুঝে সমর্থনমূলক এবং সহানুভতিপূর্ণ উত্তর দিন।

<|user|>
Topic: {example['Topics']}
Title: {example['Question-Title']}
Question: {example['Questions']}

<|assistant|>
{example['Answers']}

<|end_of_text|>"""

        return {"text": text}

    def preprocess(self, dataset):
        dataset = dataset.map(self.format_prompt)
        dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

        train_dataset = dataset["train"]
        val_dataset = dataset["test"]

        return train_dataset, val_dataset

In [30]:
processor = DatasetProcessor(
    tokenizer=tuner.tokenizer,
    max_length=8192
)

In [31]:
train_dataset, val_dataset = processor.preprocess(dataset)

Map:   0%|          | 0/38233 [00:00<?, ? examples/s]

In [32]:
print(train_dataset)
print(val_dataset)

Dataset({
    features: ['Topics', 'Question-Title', 'Questions', 'Answers', 'text'],
    num_rows: 34409
})
Dataset({
    features: ['Topics', 'Question-Title', 'Questions', 'Answers', 'text'],
    num_rows: 3824
})


In [33]:
print(train_dataset[0]["text"])

<|begin_of_text|><|system|>
আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকারীর অনুভূতি বুঝে সমর্থনমূলক এবং সহানুভতিপূর্ণ উত্তর দিন।

<|user|>
Topic: আতঙ্কিত
Title: পোষা প্রাণী হিসাবে মাকড়সা
Question: আমার বন্ধুর পোষা প্রাণী হিসাবে মাকড়সার একটি গুচ্ছ আছে, কিন্তু আমি তাদের সহ্য করতে পারি না। তারা আমাকে ভয় পায়!

<|assistant|>
আমি ভয় পাচ্ছি কেউ বেরিয়ে আসবে!

<|end_of_text|>


In [34]:
train_dataset = train_dataset.remove_columns(
    ["Topics", "Question-Title", "Questions", "Answers"]
)

val_dataset = val_dataset.remove_columns(
    ["Topics", "Question-Title", "Questions", "Answers"]
)

In [35]:
print(train_dataset.column_names)

['text']


In [36]:
print(train_dataset.column_names)
print(train_dataset[0])

['text']
{'text': '<|begin_of_text|><|system|>\nআপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকারীর অনুভূতি বুঝে সমর্থনমূলক এবং সহানুভতিপূর্ণ উত্তর দিন।\n\n<|user|>\nTopic: আতঙ্কিত\nTitle: পোষা প্রাণী হিসাবে মাকড়সা\nQuestion: আমার বন্ধুর পোষা প্রাণী হিসাবে মাকড়সার একটি গুচ্ছ আছে, কিন্তু আমি তাদের সহ্য করতে পারি না। তারা আমাকে ভয় পায়!\n\n<|assistant|>\nআমি ভয় পাচ্ছি কেউ বেরিয়ে আসবে!\n\n<|end_of_text|>'}


## Training

In [38]:
tuner.train(train_dataset, val_dataset)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/34409 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/3824 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 34,409 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
10,0.676400
20,0.466300
30,0.400700
40,0.410800
50,0.416400
60,0.407800
70,0.412200
80,0.401400
90,0.390100
100,0.389800


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [41]:
trainer = tuner.get_trainer()

model = trainer.model
tokenizer = trainer.processing_class

## Generate responses on test prompts

In [42]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

prompt = """<|begin_of_text|><|user|>
Topic: দুঃখ
Title: একাকীত্ব
Question: আমি খুব একা অনুভব করছি।

<|assistant|>"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

<|user|>
Topic: দুঃখ
Title: একাকীত্ব
Question: আমি খুব একা অনুভব করছি।

<|assistant|> এটা খুব দুঃখজনক.




In [43]:
from unsloth import FastLanguageModel
import pandas as pd
import re

FastLanguageModel.for_inference(model)

sample_eval = val_dataset.shuffle(seed=42).select(range(10))

results = []

for row in sample_eval:

    prompt = row["text"].split("<|assistant|>")[0] + "<|assistant|>"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=8192
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    response = decoded.split("<|assistant|>")[-1]
    response = re.split(r"<\|.*?\|>", response)[0].strip()

    true_answer = row["text"].split("<|assistant|>")[1].split("<|end_of_text|>")[0].strip()

    results.append({
        "input_text": prompt,
        "reference_answer": true_answer,
        "response_text": response
    })

df = pd.DataFrame(results)
df.to_csv("cleaned_generated_responses.csv", index=False)

print("Saved cleaned responses.")

Saved cleaned responses.


In [44]:
import pandas as pd

df = pd.read_csv("/kaggle/working/cleaned_generated_responses.csv")
df.head()

,input_text,reference_answer,response_text
0,<|begin_of_text|><|system|>\nআপনি একজন সহানুভূ...,আমি ভাবছি আপনি সেখানে কিভাবে পেয়েছেন.,এটা সত্যিই খারাপ. আপনি একটি সুযোগ খুঁজে পেয়েছ...
1,<|begin_of_text|><|system|>\nআপনি একজন সহানুভূ...,শুধু ইতিবাচক চিন্তা ভাবুন,এটা একটা গুরুত্বপূর্ণ সিদ্ধান্ত। এটি আপনার জীব...
2,<|begin_of_text|><|system|>\nআপনি একজন সহানুভূ...,এটা চমৎকার. আমি এটা চেষ্টা করার জন্য অপেক্ষা ক...,এটা সত্যিই একটি দুর্দান্ত খবর ছিল. আমি আশা করি...
3,<|begin_of_text|><|system|>\nআপনি একজন সহানুভূ...,ওহ এটা খুবই দুঃখজনক আমি আশা করি আপনি এটি খুঁজে...,"ওহ না, এটি একটি ভাল জিনিস হতে পারে যা আপনি ভুল..."
4,<|begin_of_text|><|system|>\nআপনি একজন সহানুভূ...,আমি পাথরের মতো এবং নিজেকে রোল করি।,আমি আশা করি তারা আবার জড়িত হয়।


In [45]:
import re

df["input_text"] = df["input_text"].str.replace(r"<\|.*?\|>", "", regex=True)

df["input_text"] = df["input_text"].str.replace("\n", " ", regex=False)

df["input_text"] = df["input_text"].str.replace(r"\s+", " ", regex=True)

df["input_text"] = df["input_text"].str.strip()


df["response_text"] = df["response_text"].str.replace(r"<\|.*?\|>", "", regex=True)
df["response_text"] = df["response_text"].str.replace(r"\s+", " ", regex=True)
df["response_text"] = df["response_text"].str.strip()

In [46]:
df.to_csv("cleaned_generated_responses.csv", index=False)

In [47]:
import pandas as pd

df = pd.read_csv("/kaggle/working/cleaned_generated_responses.csv")
df.head()

,input_text,reference_answer,response_text
0,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,আমি ভাবছি আপনি সেখানে কিভাবে পেয়েছেন.,এটা সত্যিই খারাপ. আপনি একটি সুযোগ খুঁজে পেয়েছ...
1,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,শুধু ইতিবাচক চিন্তা ভাবুন,এটা একটা গুরুত্বপূর্ণ সিদ্ধান্ত। এটি আপনার জীব...
2,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,এটা চমৎকার. আমি এটা চেষ্টা করার জন্য অপেক্ষা ক...,এটা সত্যিই একটি দুর্দান্ত খবর ছিল. আমি আশা করি...
3,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,ওহ এটা খুবই দুঃখজনক আমি আশা করি আপনি এটি খুঁজে...,"ওহ না, এটি একটি ভাল জিনিস হতে পারে যা আপনি ভুল..."
4,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,আমি পাথরের মতো এবং নিজেকে রোল করি।,আমি আশা করি তারা আবার জড়িত হয়।


## Evaluation libraries

In [48]:
!pip install evaluate nltk rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


## Load evaluation data

In [49]:
import pandas as pd

df = pd.read_csv("cleaned_generated_responses.csv")

predictions = df["response_text"].tolist()

references = df["reference_answer"].tolist()

predictions = [p.replace("\n", " ").strip() for p in predictions]
references = [r.replace("\n", " ").strip() for r in references]

import re

predictions = [re.sub(r"\s+", " ", p).strip() for p in predictions]
references = [re.sub(r"\s+", " ", r).strip() for r in references]

## BLEU score

In [50]:
from nltk.translate.bleu_score import sentence_bleu

bleu_scores = []

for ref, pred in zip(references, predictions):
    score = sentence_bleu([ref.split()], pred.split())
    bleu_scores.append(score)

print("Average BLEU:", sum(bleu_scores)/len(bleu_scores))

Average BLEU: 6.41193285632829e-232


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

## ROUGE score

In [51]:
import evaluate

rouge = evaluate.load("rouge")

rouge_results = rouge.compute(
    predictions=predictions,
    references=references
)

print(rouge_results)

{'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}


## Perplexity

In [52]:
import torch
import math

def calculate_perplexity(text):

    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])

    loss = outputs.loss
    perplexity = math.exp(loss.item())

    return perplexity

ppls = []

for text in predictions:
    ppls.append(calculate_perplexity(text))

print("Average Perplexity:", sum(ppls)/len(ppls))

Average Perplexity: 2.5644892794565255


In [53]:
import pandas as pd

df = pd.read_csv("/kaggle/working/cleaned_generated_responses.csv")


eval_samples = df.sample(10, random_state=42).reset_index(drop=True)

eval_samples.head()

,input_text,reference_answer,response_text
0,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,"হ্যাঁ, এটি একটি বরং অদ্ভুত পরিস্থিতি। আমি নিশ্...",আমি বুঝতে পারি. আপনি কি করবেন?
1,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,শুধু ইতিবাচক চিন্তা ভাবুন,এটা একটা গুরুত্বপূর্ণ সিদ্ধান্ত। এটি আপনার জীব...
2,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,আমি বুঝতে পারি যে এটি সামাজিকীকরণ করতে কখনও কখ...,আমি দুঃখিত. আপনি কি এটা ব্যতীত কিছু করতে পারেন?
3,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,আমি ভাবছি আপনি সেখানে কিভাবে পেয়েছেন.,এটা সত্যিই খারাপ. আপনি একটি সুযোগ খুঁজে পেয়েছ...
4,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,"ঠিক আছে, আসুন খুব বেশি লক্ষ্য না করি। আমরা যদি...",এটা সত্যিই সত্য যে প্রায়ই আমাদের সবাই আমাদের ...


In [54]:
evaluation_table = pd.DataFrame({
    "ID": range(1, len(eval_samples) + 1),
    "Prompt": eval_samples["input_text"],
    "Model_Response": eval_samples["response_text"],
    "Empathy (1-5)": "",
    "Relevance (1-5)": "",
    "Fluency (1-5)": "",
    "Notes": ""
})

evaluation_table

,ID,Prompt,Model_Response,Empathy (1-5),Relevance (1-5),Fluency (1-5),Notes
0,1,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,আমি বুঝতে পারি. আপনি কি করবেন?,,,,
1,2,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,এটা একটা গুরুত্বপূর্ণ সিদ্ধান্ত। এটি আপনার জীব...,,,,
2,3,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,আমি দুঃখিত. আপনি কি এটা ব্যতীত কিছু করতে পারেন?,,,,
3,4,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,এটা সত্যিই খারাপ. আপনি একটি সুযোগ খুঁজে পেয়েছ...,,,,
4,5,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,এটা সত্যিই সত্য যে প্রায়ই আমাদের সবাই আমাদের ...,,,,
5,6,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,এটা সত্যিই একটি দুর্দান্ত খবর ছিল. আমি আশা করি...,,,,
6,7,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,আমি খুব খুশি যে আপনি এখনও সেখানে আছেন এবং এটি ...,,,,
7,8,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,আমি আশা করি তারা আবার জড়িত হয়।,,,,
8,9,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,"ওহ না, এটি একটি ভাল জিনিস হতে পারে যা আপনি ভুল...",,,,
9,10,আপনি একজন সহানুভূতিশীল বাংলা সহকারী। ব্যবহারকা...,"এটা খুব খারাপ, আমি নিশ্চিত যে আপনি এটি ভাল করে...",,,,


In [55]:
evaluation_table.to_csv("human_evaluation_table.csv", index=False)

## GeneratedResponses logging

In [56]:
from datetime import datetime
import pandas as pd

df = pd.read_csv("cleaned_generated_responses.csv")

df["id"] = range(len(df))
df["experiment_id"] = 1
df["timestamp"] = datetime.now()

df = df[[
    "id",
    "experiment_id",
    "input_text",
    "response_text",
    "timestamp"
]]

df.to_csv("GeneratedResponses.csv", index=False)

In [59]:
import math

val_loss = math.log(sum(ppls)/len(ppls))

print("Validation Loss:", val_loss)

Validation Loss: 0.9417593474744497


## Experiment logging

In [61]:
from datetime import datetime
import pandas as pd

train_loss = None
eval_loss = None

for log in trainer.state.log_history:
    if "loss" in log:
        train_loss = log["loss"]
    if "eval_loss" in log:
        eval_loss = log["eval_loss"]


experiment_log = {
    "id": 1,
    "model_name": "LLaMA-3.1-8B-Instruct",
    "lora_config": "r=16 alpha=32 dropout=0.05",
    "train_loss": train_loss,
    "val_loss": val_loss,
    "perplexity": sum(ppls)/len(ppls),
    "bleu": sum(bleu_scores)/len(bleu_scores),
    "rouge": rouge_results,
    "timestamp": datetime.now()
}

exp_df = pd.DataFrame([experiment_log])
exp_df.to_csv("LLAMAExperiments.csv", index=False)